# VideoTool — FULL RENDER on Kaggle **TPU** (224 vCPU, x264)

Same pipeline as `videotool-render` (GPU/NVENC), but for the **TPU runtime**. The TPU chip itself is
unused — ffmpeg has no TPU backend. What we want is the TPU VM's **host CPU**: 224 vCPU / 405 GB RAM
and ffmpeg 7.1, versus 4 vCPU / ffmpeg 4.4.2 on the GPU box.

Measured 2026-07-24 on an 8-min synthetic render (24 scenes, same filter chains as the real pipeline):

| runtime | scene clips | mux | total | x realtime |
|---|---|---|---|---|
| **TPU — x264, 224 core** | 11.8s | 55.9s | **67.7s** | **7.09x** |
| GPU — NVENC, 4 core | 31.6s | 149.0s | 180.6s | 2.66x |
| GPU — x264, 4 core | 560.5s | 664.8s | 1225.3s | 0.39x |

So: **2.67x faster than the GPU box**, and the file is ~17% smaller at higher quality (x264 CRF20
beats NVENC cq23). Render compute only — stage-in/publish is network-bound and unchanged, so expect
roughly 1.4x end-to-end on a real episode.

Quota-wise the two runtimes stack: GPU 30h/week + TPU 20h/week. Use whichever is free.

## One-time setup (do ONCE, ever)
1. On your machine: `base64 -w0 ~/.config/rclone/rclone.conf` -> copy the single line.
2. Kaggle: Add-ons -> Secrets -> add `RCLONE_CONF` = that base64 line, toggle **Attached**.
3. Right panel -> Session options -> Accelerator = **TPU** (NOT None — a CPU-only box is 4 cores and
   renders at 0.39x realtime, i.e. slower than the video is long). Cell 2 aborts if it sees < 32 cores.
4. Save Version -> **Save & Run All (Commit)**.

## Every episode after that (NO `kaggle kernels push`, so the secret stays attached)
- Claude Code CLI writes the job config to Drive. This notebook reads
  `render_job.tpu.json` if present, else falls back to the shared `render_job.json`.
  - Same episode on either runtime -> just stage `render_job.json` and open whichever kernel.
  - **Two different episodes at once** (burning both quotas in parallel) -> stage
    `render_job.tpu.json` for this one and `render_job.json` for the GPU one.
- Open THIS saved kernel and click **Save & Run All**. No secret re-toggle, no path edits.

## Resume across runtimes — READ THIS
The encoder is pinned into the checkpoint on the first run and a resume never re-probes it
(`cloud_render_runner._assert_encoder_supported`). So:
- Episode started on **GPU** (pinned `h264_nvenc-capped`) -> **cannot** resume here; it aborts by
  design rather than welding x264 clips into NVENC ones. Resume it on the GPU kernel.
- Episode started **here** (pinned `libx264-balanced-capped`) -> *can* resume on the GPU kernel, but
  it would finish on 4 cores at 0.39x realtime. Resume it here instead.

Pick a runtime per episode and stay on it.


In [ ]:
# Print the box specs FIRST: a secret/config failure later still tells us which accelerator
# Kaggle actually gave us (TPU host = 224 cores; a CPU-only or GPU box = 4).
import multiprocessing, os, shutil, subprocess
_ram = round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9, 1)
_ff = shutil.which('ffmpeg')
_ffv = subprocess.run([_ff, '-hide_banner', '-version'], capture_output=True, text=True).stdout.splitlines()[0] if _ff else 'ffmpeg MISSING'
print(f'box: {multiprocessing.cpu_count()} cores | {_ram} GB RAM | tpu_dev={os.path.exists("/dev/accel0")} | {_ffv}', flush=True)

import os, subprocess, base64
from kaggle_secrets import UserSecretsClient
try:
    raw = UserSecretsClient().get_secret('RCLONE_CONF').strip()
except Exception as e:
    raise RuntimeError(
        'RCLONE_CONF secret is not attached to this kernel. `kaggle kernels push` does NOT inherit the attachment — in the Kaggle UI open Add-ons -> Secrets -> toggle RCLONE_CONF Attached ON, then Save & Run All. (Original error: ' + repr(e) + ')')

try:
    conf = base64.b64decode(raw, validate=True).decode('utf-8'); assert '[gdrive]' in conf
except Exception:
    conf = raw
os.makedirs(os.path.expanduser('~/.config/rclone'), exist_ok=True)
open(os.path.expanduser('~/.config/rclone/rclone.conf'), 'w').write(conf)
# The TPU host has no `sudo`, so the official `curl | sudo bash` installer fails. Install rclone into
# user space (pure-Python unzip -> ~/.local/bin, on PATH) — works on the sudo-less TPU box.
import glob, zipfile, urllib.request
_bin = os.path.expanduser('~/.local/bin'); os.makedirs(_bin, exist_ok=True)
os.environ['PATH'] = _bin + os.pathsep + os.environ.get('PATH', '')
if shutil.which('rclone') is None:
    urllib.request.urlretrieve('https://downloads.rclone.org/rclone-current-linux-amd64.zip', '/tmp/rclone.zip')
    with zipfile.ZipFile('/tmp/rclone.zip') as _zf:
        _zf.extractall('/tmp/rclone-dl')
    shutil.copy(glob.glob('/tmp/rclone-dl/rclone-*-linux-amd64/rclone')[0], f'{_bin}/rclone')
    os.chmod(f'{_bin}/rclone', 0o755)
remotes = subprocess.run(['rclone', 'listremotes'], capture_output=True, text=True).stdout.strip()
print('rclone remotes:', remotes or '(NONE)', '| conf has [gdrive]:', '[gdrive]' in conf)
assert 'gdrive:' in remotes.split(), 'RCLONE_CONF has no [gdrive] remote (set it to base64 of rclone.conf).'
SHARED = 'gdrive:_VIDEOTOOL_SHARED'
for mod in ('videotool_cloud.py', 'cloud_director.py', 'cloud_render_runner.py'):
    subprocess.run(['rclone', 'copyto', f'{SHARED}/{mod}', mod], check=True)
for lib in ('sfx', 'overlays'):
    dst = os.path.expanduser(f'~/.local/share/videotool/{lib}')
    os.makedirs(dst, exist_ok=True)
    subprocess.run(['rclone', 'copy', f'{SHARED}/{lib}', dst, '--fast-list'], check=False)
import cloud_render_runner as rr
print('setup OK -> ready to render')


In [ ]:
# TPU variant of the GPU render cell: no NVENC on this box, so x264 on many cores instead.
# Reads render_job.tpu.json if staged (lets a TPU episode run in parallel with a GPU one),
# else the shared render_job.json.
import json, os, subprocess

CONFIG_REMOTES = ('gdrive:_VIDEOTOOL_SHARED/render_job.tpu.json',
                  'gdrive:_VIDEOTOOL_SHARED/render_job.json')
raw, used = '', None
for remote in CONFIG_REMOTES:
    raw = subprocess.run(['rclone', 'cat', remote], capture_output=True, text=True).stdout.strip()
    if raw:
        used = remote
        break
if not raw:
    raise RuntimeError(
        'No render job config at ' + ' or '.join(CONFIG_REMOTES) + '. The Claude Code CLI writes it '
        'per episode (a small JSON with source/output/checkpoint[/creative]). Nothing to render.')
cfg = json.loads(raw)
for k in ('source', 'output', 'checkpoint'):
    if not cfg.get(k):
        raise RuntimeError(f'{used} is missing required key: {k!r}')

# A CPU-only session looks identical to this notebook until the render crawls, so fail loudly now.
# Measured: 4-core x264 runs at 0.39x realtime; the TPU host has 224.
cores = os.cpu_count() or 1
if cores < 32:
    raise RuntimeError(
        f'Only {cores} CPU cores visible — this is not the TPU runtime. This notebook renders on CPU '
        '(x264), so a 4-core box would run at ~0.39x realtime. Right panel -> Session options -> '
        'Accelerator = TPU, then Save & Run All. (To render on the GPU box use the videotool-render '
        'kernel, which uses NVENC.)')

# One ffmpeg per scene clip; the default caps at 8 (fine for a 4-core box, wasteful on 224).
os.environ['VIDEOTOOL_SCENE_WORKERS'] = str(cfg.get('scene_workers', 32))

print('config:', used)
print('Rendering:', cfg['source'])
print('     -> output:', cfg['output'])
print('     -> checkpoint:', cfg['checkpoint'])
print(f'box: {cores} cores | scene workers: ' + os.environ['VIDEOTOOL_SCENE_WORKERS'])
rr.render_job(
    cfg['source'], cfg['output'], cfg['checkpoint'],
    creative_remote=cfg.get('creative'),
    repo_ref=cfg.get('repo_ref') or 'git+https://github.com/pnd4189/video-tool@main',
    allow_cpu=True,  # no NVENC on the TPU host -> probe_encoder falls back to libx264-balanced-capped
    local_job='/tmp/job',
)
